# The K-Suiter: patient-specific EEG channel ranking

Given a patient and a fixed `K`, this notebook ranks and returns the `K` EEG channels that best suit that patient's seizure-forecasting history.

The ranking uses expanding chronological validation: models learn from earlier seizure episodes and validate on later episodes. At each step, every remaining channel is tested alongside the channels already selected. The candidate with the highest value is added:

$$\text{value} = \text{validation AUPRC} - 0.25 \times \text{validation Brier score}$$

AUPRC is emphasized because seizure targets are imbalanced; the Brier penalty discourages poorly calibrated probabilities. The output is nested and reproducible: the first two channels for `K=4` are the same channels returned for `K=2`.

> **Research only:** this ranking has not been clinically validated and must not directly control patient care.

## 1. Inputs

Set the patient identifier and number of desired channels. Siena identifiers look like `PN00`, `PN06`, and `PN10`.

In [ ]:
PATIENT_ID = "PN00"
K = 4
FORCE_REBUILD_FEATURES = False

## 2. Setup

The notebook can be launched from either the scripts directory or the repository root. Feature extraction is cached; the first run for a patient may take several minutes.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "k_suiter.py").exists():
    candidates = list(NOTEBOOK_DIR.rglob("k_suiter.py"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Run this notebook from the scripts directory or repository root."
        )
    NOTEBOOK_DIR = candidates[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import rolling_seizure_forecasting as rsf
import personalized_channels_workflow as pc
from k_suiter import KSuiter

paths = pc.personalized_paths(NOTEBOOK_DIR)
forecast_config = rsf.ForecastConfig(test_fraction=0.20, max_iter=60)
suiter_config = pc.PersonalizedConfig(
    k=K,
    patient_ids=(PATIENT_ID,),
    swap_refinement=False,
    force_rebuild_features=FORCE_REBUILD_FEATURES,
)
suiter_config.validate()
print(f"Patient={PATIENT_ID}; K={K}")

## 3. Load one patient's channel-local features

Eligibility requires at least two usable historical seizure events and at least `K` consistently available channels. Only channel-local features are used, so an excluded electrode cannot leak information into a selected electrode.

In [ ]:
manifest = pc.load_manifest(paths, forecast_config)
available_patients = sorted(manifest["patient_id"].astype(str).unique())
if PATIENT_ID not in available_patients:
    raise ValueError(
        f"Unknown patient {PATIENT_ID!r}. Available patients: {available_patients}"
    )

patient_manifest = manifest.loc[
    manifest["patient_id"].astype(str).eq(PATIENT_ID)
].copy()
n_events = patient_manifest.loc[
    patient_manifest["episode_type"].eq("preictal"), "source_event_id"
].nunique()
if n_events < 2:
    raise ValueError(
        f"{PATIENT_ID} has {n_events} usable seizure event(s); at least 2 are required."
    )

patient = pc.build_patient_feature_data(
    patient_manifest,
    paths["feature_cache"],
    forecast_config,
    force=FORCE_REBUILD_FEATURES,
)
patient_overview = pd.DataFrame(
    {
        "patient_id": [patient.patient_id],
        "usable_seizure_events": [n_events],
        "available_channels": [len(patient.channel_names)],
        "landmark_rows": [len(patient.frame)],
    }
)
display(patient_overview)

## 4. Run the K-Suiter

The first result is the requested simple output: the ordered channel names. The table adds audit details. `marginal_value` measures the change in the complete selected set's value after adding that channel; it need not always be positive.

In [ ]:
suiter = KSuiter(config=suiter_config)
selected_channels = suiter.recommend(patient, K)

print(f"Top {K} channels for {PATIENT_ID}: {selected_channels}")
display(suiter.ranking_.style.format(
    {
        "value": "{:.4f}",
        "marginal_value": "{:.4f}",
        "validation_auprc": "{:.4f}",
        "validation_brier": "{:.4f}",
    }
))

## 5. Interpretation

- Rank 1 is the strongest channel when used alone.
- Each later channel is the best addition conditional on the channels above it.
- Validation is chronological and patient-specific; results should not be interpreted as a universal electrode ranking.
- A high validation score on a patient with few events is uncertain. Report the event count with every ranking.
- The selected channels should be evaluated on a later untouched episode before making research performance claims.